In [ ]:
# ============================================================
# EXP 9: ECG Classification using KaggleHub Dataset
# ============================================================

# ============================================================
# STEP 1: Import Libraries
# ============================================================
import pandas as pd
import numpy as np
import os

import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# ============================================================
# STEP 2: Download Dataset using KaggleHub
# ============================================================
path = kagglehub.dataset_download("shayanfazeli/heartbeat")
print("Dataset downloaded at:", path)

100%|██████████| 98.8M/98.8M [00:03<00:00, 33.2MB/s]

Extracting files...


Dataset downloaded at: /root/.cache/kagglehub/datasets/shayanfazeli/heartbeat/versions/1


In [ ]:
# ============================================================
# STEP 3: Load Dataset Files
# ============================================================
# This dataset contains:
# mitbih_train.csv
# mitbih_test.csv
# ptbdb_normal.csv
# ptbdb_abnormal.csv

files = os.listdir(path)
print("Files in dataset:", files)

# We will use PTB dataset (binary classification: normal vs abnormal)
normal_path = os.path.join(path, "ptbdb_normal.csv")
abnormal_path = os.path.join(path, "ptbdb_abnormal.csv")

normal = pd.read_csv(normal_path, header=None)
abnormal = pd.read_csv(abnormal_path, header=None)

# Combine both datasets
data = pd.concat([normal, abnormal], axis=0)

print("Dataset Shape:", data.shape)

Files in dataset: ['ptbdb_abnormal.csv', 'mitbih_test.csv', 'ptbdb_normal.csv', 'mitbih_train.csv']
Dataset Shape: (14552, 188)


In [ ]:
# ============================================================
# STEP 4: Data Preprocessing
# ============================================================
# Last column = label (0 = normal, 1 = abnormal)

X = data.iloc[:, :-1]
y = data.iloc[:, -1]

print("Feature Shape:", X.shape)
print("Target Distribution:\n", y.value_counts())

Feature Shape: (14552, 187)
Target Distribution:
 187
1.0    10506
0.0     4046
Name: count, dtype: int64


In [ ]:
# ============================================================
# STEP 5: Train-Test Split
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# ============================================================
# STEP 6: Define ML Pipelines
# ============================================================
# Pipeline = Scaling + Model

pipelines = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ]),

    "Random Forest": Pipeline([
        ("scaler", StandardScaler()),   # optional but kept for consistency
        ("model", RandomForestClassifier(n_estimators=100))
    ]),

    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC())
    ]),

    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier())
    ]),

    "Decision Tree": Pipeline([
        ("scaler", StandardScaler()),
        ("model", DecisionTreeClassifier())
    ])
}

In [ ]:
# ============================================================
# STEP 7: Train and Evaluate Models
# ============================================================
results = {}

for name, pipe in pipelines.items():
    print("\n=================================")
    print(f"Model: {name}")
    print("=================================")

    # Train
    pipe.fit(X_train, y_train)

    # Predict
    y_pred = pipe.predict(X_test)

    # Accuracy
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc

    print("Accuracy:", round(acc, 4))

    # Classification report
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # Confusion matrix
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))


Model: Logistic Regression
Accuracy: 0.8282

Classification Report:
              precision    recall  f1-score   support

         0.0       0.75      0.57      0.65       809
         1.0       0.85      0.93      0.89      2102

    accuracy                           0.83      2911
   macro avg       0.80      0.75      0.77      2911
weighted avg       0.82      0.83      0.82      2911

Confusion Matrix:
[[ 458  351]
 [ 149 1953]]

Model: Random Forest
Accuracy: 0.9705

Classification Report:
              precision    recall  f1-score   support

         0.0       0.97      0.92      0.95       809
         1.0       0.97      0.99      0.98      2102

    accuracy                           0.97      2911
   macro avg       0.97      0.95      0.96      2911
weighted avg       0.97      0.97      0.97      2911

Confusion Matrix:
[[ 743   66]
 [  20 2082]]

Model: SVM
Accuracy: 0.9234

Classification Report:
              precision    recall  f1-score   support

         0.0    

In [ ]:
# ============================================================
# STEP 8: Model Comparison
# ============================================================
print("\n=================================")
print("Final Model Comparison")
print("=================================")

for model, acc in results.items():
    print(f"{model}: {acc:.4f}")

best_model = max(results, key=results.get)
print("\nBest Performing Model:", best_model)


Final Model Comparison
Logistic Regression: 0.8282
Random Forest: 0.9705
SVM: 0.9234
KNN: 0.9200
Decision Tree: 0.9172

Best Performing Model: Random Forest
